In [3]:
# ============================================================
# FINAL CLEAN VERSION (YOUR CODE — FULLY WORKING)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ================= CONFIG =================
N=480; SZ=80; K=15; M=64
SNR=12; N_TRAIN=2000; N_TEST=200; BSZ=32
DEC_THR = 0.1

# ================= DATA =================

def make_uwb_signals(n, seed):
    rng = np.random.default_rng(seed)
    S = np.zeros((n, N), np.float32)

    for i in range(n):
        centres = rng.choice(SZ - 6, K, replace=False) + 3
        for c in centres:
            amp = rng.standard_normal()
            width = rng.uniform(1.5, 3.0)
            for d in range(-4, 5):
                if 0 <= c+d < N:
                    S[i, c+d] += amp * np.exp(-0.5*(d/width)**2)

    S /= (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
    return S


def add_noise(Y, snr_db):
    sp = (Y**2).mean(1, keepdims=True)
    std = np.sqrt(sp / 10**(snr_db/10))
    return Y + np.random.randn(*Y.shape).astype(np.float32) * std


PHI = np.random.randn(M, N).astype(np.float32) / np.sqrt(M)

S_tr = make_uwb_signals(N_TRAIN, 1)
Y_tr = add_noise((PHI @ S_tr.T).T, SNR)

S_te = make_uwb_signals(N_TEST, 3)
Y_te = add_noise((PHI @ S_te.T).T, SNR)

# ================= SEARCH DATA =================

def make_search_data(S):
    n = S.shape[0]
    R = np.zeros((n, K, M), np.float32)
    U = np.zeros((n, K), np.int64)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=PHI@h; r=y.copy(); sel=[]

        for k in range(K):
            R[z,k]=r
            idx=int(np.argmax(np.abs(s[:SZ])))
            U[z,k]=idx

            s[idx]=0
            sel.append(idx)

            r=y-PHI[:,sel]@h[sel]

    return R,U


R_tr, U_tr = make_search_data(S_tr)

class SearchDS(Dataset):
    def __init__(self, R, U):
        self.R=torch.tensor(R)
        self.U=torch.tensor(U)

    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i], self.U[i]

# ================= LSTM =================

class LSTMCell(nn.Module):
    def __init__(self, I,H):
        super().__init__()
        self.Wxi = nn.Linear(I,H)
        self.Whi = nn.Linear(H,H,bias=False)
        self.Wxg = nn.Linear(I,H)
        self.Whg = nn.Linear(H,H,bias=False)
        self.Wxf = nn.Linear(I,H)
        self.Whf = nn.Linear(H,H,bias=False)
        self.Wxo = nn.Linear(I,H)
        self.Who = nn.Linear(H,H,bias=False)

    def forward(self,x,h,c):
        i=torch.sigmoid(self.Wxi(x)+self.Whi(h))
        g=torch.tanh(self.Wxg(x)+self.Whg(h))
        f=torch.sigmoid(self.Wxf(x)+self.Whf(h))
        o=torch.sigmoid(self.Wxo(x)+self.Who(h))
        c=f*c+i*g
        h=o*torch.tanh(c)
        return h,c


class SearchNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.H=512
        self.cell=LSTMCell(M,self.H)
        self.out=nn.Linear(self.H,N,bias=False)

    def forward(self,x):
        B,T,_=x.shape
        h=torch.zeros(B,self.H,device=x.device)
        c=torch.zeros(B,self.H,device=x.device)
        outs=[]

        for t in range(T):
            h,c=self.cell(x[:,t],h,c)
            outs.append(self.out(h).unsqueeze(1))

        return torch.cat(outs,dim=1)


search_net=SearchNet().to(DEVICE)

# ================= DECISION NETWORK =================

class DecisionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(N,256),
            nn.ReLU(),
            nn.Linear(256,N),
            nn.Sigmoid()
        )

    def forward(self,x): return self.net(x)


dec_net=DecisionNet().to(DEVICE)

idx_input=torch.eye(N).to(DEVICE)
labels=torch.tensor((S_tr!=0).mean(0),dtype=torch.float32).unsqueeze(0).repeat(N,1).to(DEVICE)

opt=optim.Adam(dec_net.parameters(),lr=1e-2)

for ep in range(150):
    pred=dec_net(idx_input)
    loss=nn.BCELoss()(pred,labels)
    opt.zero_grad(); loss.backward(); opt.step()

dec_net.eval()
with torch.no_grad():
    dec_prob=dec_net(idx_input)[0].cpu().numpy()

# ================= TRAIN SEARCH =================

dl=DataLoader(SearchDS(R_tr,U_tr),BSZ,shuffle=True)
opt=optim.Adam(search_net.parameters(),lr=1e-2)
loss_fn=nn.CrossEntropyLoss()

for ep in range(30):
    for Rb,Ub in dl:
        Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
        logits=search_net(Rb)
        loss=loss_fn(logits.reshape(-1,N),Ub.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    print("epoch",ep,"loss",loss.item())

search_net.eval()

# ================= SFDLCS =================

def sfdlcs(y):
    r=y.copy()
    hist=[]
    sel=[]

    for k in range(K):
        hist.append(r.copy())

        seq=torch.tensor(np.stack(hist),dtype=torch.float32)\
             .unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits=search_net(seq)[0,-1]

        probs=torch.softmax(logits,dim=-1).cpu().numpy()

        order=np.argsort(probs)[::-1]

        chosen=None
        for cand in order:
            if cand in sel: continue
            if dec_prob[cand]>DEC_THR:
                chosen=cand; break

        if chosen is None: break

        sel.append(chosen)

        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta

    s_hat=np.zeros(N)
    if sel:
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]

    return s_hat

# ================= METRICS =================

def nmse(t,p):
    return np.mean(np.linalg.norm(t-p,axis=1)**2 /
                   (np.linalg.norm(t,axis=1)**2+1e-12))

def crr(t,p):
    vals=[]
    for s,s_hat in zip(t,p):
        nz=np.where(s!=0)[0]
        if len(nz)==0: continue
        vals.append(np.mean(np.abs(s[nz]-s_hat[nz])<0.1))
    return np.mean(vals)

# ================= EVAL =================

print("\n=== EVALUATION ===")

preds=[]
t0=time.time()

for i in range(N_TEST):
    preds.append(sfdlcs(Y_te[i]))

preds=np.array(preds)

print("NMSE:",nmse(S_te,preds))
print("CRR :",crr(S_te,preds))
print("Time per sample:",(time.time()-t0)/N_TEST)

epoch 0 loss 4.285009860992432
epoch 1 loss 3.8326072692871094
epoch 2 loss 3.6962504386901855
epoch 3 loss 3.299776792526245
epoch 4 loss 3.0284345149993896
epoch 5 loss 2.9053640365600586
epoch 6 loss 2.841614246368408
epoch 7 loss 2.4748454093933105
epoch 8 loss 2.406144857406616
epoch 9 loss 2.238316059112549
epoch 10 loss 2.0430688858032227
epoch 11 loss 2.0355241298675537
epoch 12 loss 1.9265449047088623
epoch 13 loss 1.6234158277511597
epoch 14 loss 1.6716123819351196
epoch 15 loss 1.5233639478683472
epoch 16 loss 1.3391265869140625
epoch 17 loss 1.3080962896347046
epoch 18 loss 1.1268088817596436
epoch 19 loss 1.1109751462936401
epoch 20 loss 0.9489717483520508
epoch 21 loss 0.911795437335968
epoch 22 loss 0.726385772228241
epoch 23 loss 0.5965566039085388
epoch 24 loss 0.6063199639320374
epoch 25 loss 0.5501482486724854
epoch 26 loss 0.3979010283946991
epoch 27 loss 0.33792322874069214
epoch 28 loss 0.34416982531547546
epoch 29 loss 0.2903992235660553

=== EVALUATION ===
NMSE:

In [4]:
# ================= OMP =================

def omp(y):
    r = y.copy()
    sel = []

    for _ in range(K):
        corr = np.abs(PHI.T @ r)
        idx = int(np.argmax(corr))

        if idx in sel:
            break

        sel.append(idx)

        A = PHI[:, sel]
        theta = np.linalg.lstsq(A, y, rcond=None)[0]
        r = y - A @ theta

        if np.linalg.norm(r) < 1e-6:
            break

    s_hat = np.zeros(N)
    if sel:
        A = PHI[:, sel]
        theta = np.linalg.lstsq(A, y, rcond=None)[0]
        for i, idx in enumerate(sel):
            s_hat[idx] = theta[i]

    return s_hat

In [5]:
print("\n=== COMPARISON: SFDLCS vs OMP ===")

# ---------- SFDLCS ----------
t0 = time.time()
sf_preds = np.array([sfdlcs(Y_te[i]) for i in range(N_TEST)])
t_sf = (time.time() - t0) / N_TEST

# ---------- OMP ----------
t0 = time.time()
omp_preds = np.array([omp(Y_te[i]) for i in range(N_TEST)])
t_omp = (time.time() - t0) / N_TEST

# ---------- Metrics ----------
sf_nmse = nmse(S_te, sf_preds)
sf_crr  = crr(S_te, sf_preds)

omp_nmse = nmse(S_te, omp_preds)
omp_crr  = crr(S_te, omp_preds)

# ---------- Print ----------
print("\nAlgorithm     NMSE       CRR        Time/sample")
print("--------------------------------------------------")
print(f"SFDLCS     {sf_nmse:.4f}    {sf_crr:.4f}    {t_sf:.5f}s")
print(f"OMP        {omp_nmse:.4f}    {omp_crr:.4f}    {t_omp:.5f}s")


=== COMPARISON: SFDLCS vs OMP ===

Algorithm     NMSE       CRR        Time/sample
--------------------------------------------------
SFDLCS     0.5199    0.7574    0.05968s
OMP        1.5831    0.6380    0.00144s


In [1]:
# ============================================================
# 🔥 FULL PIPELINE: RANDOM Φ vs LEARNED Φ (SFDLCS + OMP)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time

# ================= SETUP =================
torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ================= CONFIG =================
N=480; SZ=80; K=15; M=64
SNR=12; N_TRAIN=2000; N_TEST=200; BSZ=32
DEC_THR = 0.1

# ============================================================
# ================= DATA GENERATION =================
# ============================================================

def make_uwb_signals(n, seed):
    rng = np.random.default_rng(seed)
    S = np.zeros((n, N), np.float32)

    for i in range(n):
        centres = rng.choice(SZ - 6, K, replace=False) + 3
        for c in centres:
            amp = rng.standard_normal()
            width = rng.uniform(1.5, 3.0)
            for d in range(-4, 5):
                if 0 <= c+d < N:
                    S[i, c+d] += amp * np.exp(-0.5*(d/width)**2)

    S /= (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
    return S


def add_noise(Y, snr_db):
    sp = (Y**2).mean(1, keepdims=True)
    std = np.sqrt(sp / 10**(snr_db/10))
    return Y + np.random.randn(*Y.shape).astype(np.float32) * std


# ============================================================
# 🔥 LEARNABLE Φ TRAINING
# ============================================================

class PhiLearner(nn.Module):
    def __init__(self):
        super().__init__()
        self.phi = nn.Parameter(torch.randn(M, N) / np.sqrt(M))

    def forward(self, s):
        return torch.matmul(s, self.phi.t())


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(M, 256),
            nn.ReLU(),
            nn.Linear(256, N)
        )

    def forward(self, y):
        return self.net(y)


def train_learned_phi(S_tr):
    phi_model = PhiLearner().to(DEVICE)
    decoder = Decoder().to(DEVICE)

    opt = optim.Adam(list(phi_model.parameters()) + list(decoder.parameters()), lr=1e-3)
    loss_fn = nn.MSELoss()

    S_tensor = torch.tensor(S_tr).to(DEVICE)

    print("\n=== Training Learnable Φ ===")
    for ep in range(200):
        y = phi_model(S_tensor)
        s_hat = decoder(y)

        loss = loss_fn(s_hat, S_tensor)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if ep % 20 == 0:
            print("Epoch", ep, "Loss:", loss.item())

    return phi_model.phi.detach().cpu().numpy()


# ============================================================
# ================= SEARCH DATA =================
# ============================================================

def make_search_data(S, PHI):
    n = S.shape[0]
    R = np.zeros((n, K, M), np.float32)
    U = np.zeros((n, K), np.int64)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=PHI@h; r=y.copy(); sel=[]

        for k in range(K):
            R[z,k]=r
            idx=int(np.argmax(np.abs(s[:SZ])))
            U[z,k]=idx

            s[idx]=0
            sel.append(idx)

            r=y-PHI[:,sel]@h[sel]

    return R,U


class SearchDS(Dataset):
    def __init__(self, R, U):
        self.R=torch.tensor(R)
        self.U=torch.tensor(U)

    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i], self.U[i]


# ============================================================
# ================= NETWORKS =================
# ============================================================

class LSTMCell(nn.Module):
    def __init__(self, I,H):
        super().__init__()
        self.Wxi = nn.Linear(I,H)
        self.Whi = nn.Linear(H,H,bias=False)
        self.Wxg = nn.Linear(I,H)
        self.Whg = nn.Linear(H,H,bias=False)
        self.Wxf = nn.Linear(I,H)
        self.Whf = nn.Linear(H,H,bias=False)
        self.Wxo = nn.Linear(I,H)
        self.Who = nn.Linear(H,H,bias=False)

    def forward(self,x,h,c):
        i=torch.sigmoid(self.Wxi(x)+self.Whi(h))
        g=torch.tanh(self.Wxg(x)+self.Whg(h))
        f=torch.sigmoid(self.Wxf(x)+self.Whf(h))
        o=torch.sigmoid(self.Wxo(x)+self.Who(h))
        c=f*c+i*g
        h=o*torch.tanh(c)
        return h,c


class SearchNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.H=512
        self.cell=LSTMCell(M,self.H)
        self.out=nn.Linear(self.H,N,bias=False)

    def forward(self,x):
        B,T,_=x.shape
        h=torch.zeros(B,self.H,device=x.device)
        c=torch.zeros(B,self.H,device=x.device)
        outs=[]

        for t in range(T):
            h,c=self.cell(x[:,t],h,c)
            outs.append(self.out(h).unsqueeze(1))

        return torch.cat(outs,dim=1)


class DecisionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(N,256),
            nn.ReLU(),
            nn.Linear(256,N),
            nn.Sigmoid()
        )

    def forward(self,x): return self.net(x)


# ============================================================
# ================= TRAIN FUNCTION =================
# ============================================================

def train_sfdlcs(PHI, S_tr):

    R_tr, U_tr = make_search_data(S_tr, PHI)

    search_net=SearchNet().to(DEVICE)
    dec_net=DecisionNet().to(DEVICE)

    # ---- train decision net ----
    idx_input=torch.eye(N).to(DEVICE)
    labels=torch.tensor((S_tr!=0).mean(0),dtype=torch.float32)\
            .unsqueeze(0).repeat(N,1).to(DEVICE)

    opt=optim.Adam(dec_net.parameters(),lr=1e-2)

    for _ in range(100):
        pred=dec_net(idx_input)
        loss=nn.BCELoss()(pred,labels)
        opt.zero_grad(); loss.backward(); opt.step()

    dec_net.eval()
    with torch.no_grad():
        dec_prob=dec_net(idx_input)[0].cpu().numpy()

    # ---- train search net ----
    dl=DataLoader(SearchDS(R_tr,U_tr),BSZ,shuffle=True)
    opt=optim.Adam(search_net.parameters(),lr=1e-2)
    loss_fn=nn.CrossEntropyLoss()

    for ep in range(20):
        for Rb,Ub in dl:
            Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
            logits=search_net(Rb)
            loss=loss_fn(logits.reshape(-1,N),Ub.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()

    search_net.eval()

    return search_net, dec_prob


# ============================================================
# ================= SFDLCS + OMP =================
# ============================================================

def build_solvers(PHI, search_net, dec_prob):

    def sfdlcs(y):
        r=y.copy(); hist=[]; sel=[]

        for _ in range(K):
            hist.append(r.copy())
            seq=torch.tensor(np.stack(hist),dtype=torch.float32)\
                 .unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                logits=search_net(seq)[0,-1]

            probs=torch.softmax(logits,dim=-1).cpu().numpy()
            order=np.argsort(probs)[::-1]

            chosen=None
            for cand in order:
                if cand in sel: continue
                if dec_prob[cand]>DEC_THR:
                    chosen=cand; break

            if chosen is None: break

            sel.append(chosen)
            A=PHI[:,sel]
            theta=np.linalg.lstsq(A,y,rcond=None)[0]
            r=y-A@theta

        s_hat=np.zeros(N)
        if sel:
            A=PHI[:,sel]
            theta=np.linalg.lstsq(A,y,rcond=None)[0]
            for i,idx in enumerate(sel):
                s_hat[idx]=theta[i]

        return s_hat


    def omp(y):
        r=y.copy(); sel=[]

        for _ in range(K):
            corr=np.abs(PHI.T@r)
            idx=int(np.argmax(corr))
            if idx in sel: break
            sel.append(idx)

            A=PHI[:,sel]
            theta=np.linalg.lstsq(A,y,rcond=None)[0]
            r=y-A@theta

        s_hat=np.zeros(N)
        if sel:
            A=PHI[:,sel]
            theta=np.linalg.lstsq(A,y,rcond=None)[0]
            for i,idx in enumerate(sel):
                s_hat[idx]=theta[i]

        return s_hat

    return sfdlcs, omp


# ============================================================
# ================= METRICS =================
# ============================================================

def nmse(t,p):
    return np.mean(np.linalg.norm(t-p,axis=1)**2 /
                   (np.linalg.norm(t,axis=1)**2+1e-12))

def crr(t,p):
    vals=[]
    for s,s_hat in zip(t,p):
        nz=np.where(s!=0)[0]
        if len(nz)==0: continue
        vals.append(np.mean(np.abs(s[nz]-s_hat[nz])<0.1))
    return np.mean(vals)


# ============================================================
# ================= MAIN PIPELINE =================
# ============================================================

S_tr = make_uwb_signals(N_TRAIN, 1)
S_te = make_uwb_signals(N_TEST, 3)

# ---------- CASE 1: RANDOM Φ ----------
print("\n===== RANDOM Φ =====")
PHI_rand = np.random.randn(M, N).astype(np.float32) / np.sqrt(M)

Y_tr = add_noise((PHI_rand @ S_tr.T).T, SNR)
Y_te = add_noise((PHI_rand @ S_te.T).T, SNR)

search_net, dec_prob = train_sfdlcs(PHI_rand, S_tr)
sfdlcs_fn, omp_fn = build_solvers(PHI_rand, search_net, dec_prob)

t0=time.time()
sf_preds = np.array([sfdlcs_fn(Y_te[i]) for i in range(N_TEST)])
t_sf=(time.time()-t0)/N_TEST

t0=time.time()
omp_preds = np.array([omp_fn(Y_te[i]) for i in range(N_TEST)])
t_omp=(time.time()-t0)/N_TEST

print("\nRANDOM Φ RESULTS")
print("SFDLCS NMSE:",nmse(S_te,sf_preds),"CRR:",crr(S_te,sf_preds),"Time:",t_sf)
print("OMP    NMSE:",nmse(S_te,omp_preds),"CRR:",crr(S_te,omp_preds),"Time:",t_omp)


# ---------- CASE 2: LEARNED Φ ----------
print("\n===== LEARNED Φ =====")
PHI_learned = train_learned_phi(S_tr)

Y_tr = add_noise((PHI_learned @ S_tr.T).T, SNR)
Y_te = add_noise((PHI_learned @ S_te.T).T, SNR)

search_net, dec_prob = train_sfdlcs(PHI_learned, S_tr)
sfdlcs_fn, omp_fn = build_solvers(PHI_learned, search_net, dec_prob)

t0=time.time()
sf_preds = np.array([sfdlcs_fn(Y_te[i]) for i in range(N_TEST)])
t_sf=(time.time()-t0)/N_TEST

t0=time.time()
omp_preds = np.array([omp_fn(Y_te[i]) for i in range(N_TEST)])
t_omp=(time.time()-t0)/N_TEST

print("\nLEARNED Φ RESULTS")
print("SFDLCS NMSE:",nmse(S_te,sf_preds),"CRR:",crr(S_te,sf_preds),"Time:",t_sf)
print("OMP    NMSE:",nmse(S_te,omp_preds),"CRR:",crr(S_te,omp_preds),"Time:",t_omp)


===== RANDOM Φ =====

RANDOM Φ RESULTS
SFDLCS NMSE: 0.5137657593529107 CRR: 0.7615896455214558 Time: 0.06744069457054139
OMP    NMSE: 1.583053558050754 CRR: 0.6379655348796889 Time: 0.0014926600456237793

===== LEARNED Φ =====

=== Training Learnable Φ ===
Epoch 0 Loss: 0.005253802519291639
Epoch 20 Loss: 0.0014620275469496846
Epoch 40 Loss: 0.0006028311909176409
Epoch 60 Loss: 0.0002995365939568728
Epoch 80 Loss: 0.00017825212853495032
Epoch 100 Loss: 0.00012183049693703651
Epoch 120 Loss: 8.936529775382951e-05
Epoch 140 Loss: 6.866211333544925e-05
Epoch 160 Loss: 5.484227222041227e-05
Epoch 180 Loss: 4.550122321234085e-05

LEARNED Φ RESULTS
SFDLCS NMSE: 0.4822164704050897 CRR: 0.7745453774264758 Time: 0.06848850965499878
OMP    NMSE: 1.587175195443386 CRR: 0.6394716125410654 Time: 0.0015668821334838866


In [2]:
# ============================================================
# 🔥 RESULT COMPARISON (SINGLE RUN)
# ============================================================

def compare_results(tag, sf_nmse, sf_crr, omp_nmse, omp_crr):
    print(f"\n=== {tag} ===")
    print("SFDLCS -> NMSE:", round(sf_nmse,6), " CRR:", round(sf_crr,6))
    print("OMP    -> NMSE:", round(omp_nmse,6), " CRR:", round(omp_crr,6))


# Save results from your runs
rand_sf_nmse = 0.5137657593529107
rand_sf_crr  = 0.7615896455214558
rand_omp_nmse = 1.583053558050754
rand_omp_crr  = 0.6379655348796889

learn_sf_nmse = 0.4822164704050897
learn_sf_crr  = 0.7745453774264758
learn_omp_nmse = 1.587175195443386
learn_omp_crr  = 0.6394716125410654

compare_results("RANDOM Φ", rand_sf_nmse, rand_sf_crr, rand_omp_nmse, rand_omp_crr)
compare_results("LEARNED Φ", learn_sf_nmse, learn_sf_crr, learn_omp_nmse, learn_omp_crr)


# ============================================================
# 🔥 IMPROVEMENT METRICS
# ============================================================

nmse_improve = (rand_sf_nmse - learn_sf_nmse) / rand_sf_nmse * 100
crr_improve  = (learn_sf_crr - rand_sf_crr) / rand_sf_crr * 100

print("\n=== IMPROVEMENT ===")
print(f"NMSE Improvement: {nmse_improve:.2f}%")
print(f"CRR  Improvement: {crr_improve:.2f}%")


=== RANDOM Φ ===
SFDLCS -> NMSE: 0.513766  CRR: 0.76159
OMP    -> NMSE: 1.583054  CRR: 0.637966

=== LEARNED Φ ===
SFDLCS -> NMSE: 0.482216  CRR: 0.774545
OMP    -> NMSE: 1.587175  CRR: 0.639472

=== IMPROVEMENT ===
NMSE Improvement: 6.14%
CRR  Improvement: 1.70%


In [3]:
# ============================================================
# 🔥 PRINT INITIAL RANDOM Φ
# ============================================================

print("\n=== INITIAL RANDOM Φ (sample) ===")
print(PHI_rand[:5, :10])   # first 5 rows, 10 columns

print("\nStats:")
print("Mean :", PHI_rand.mean())
print("Std  :", PHI_rand.std())
print("Min  :", PHI_rand.min())
print("Max  :", PHI_rand.max())


=== INITIAL RANDOM Φ (sample) ===
[[ 0.22050655  0.05001965  0.12234225  0.28011164  0.23344475 -0.12215973
   0.11876106 -0.01891965 -0.01290236  0.05132481]
 [ 0.09642574  0.12867986 -0.1135954  -0.0530397   0.1078245  -0.33195239
   0.18916601  0.06914151 -0.005713    0.02756346]
 [-0.04569389  0.11726157  0.03709165  0.10374827 -0.06201279 -0.00935062
   0.001529    0.19615746  0.08630363  0.09958401]
 [ 0.0184293  -0.12218311  0.10992374  0.07942807  0.06782635  0.08949236
  -0.37432662  0.1101172   0.22601648  0.05457981]
 [ 0.17381674 -0.20516858 -0.01937948  0.00825753 -0.06197444  0.15207222
  -0.04233527  0.25434533  0.13177224  0.11885421]]

Stats:
Mean : -0.000635589243866678
Std  : 0.12424240411352011
Min  : -0.5824941396713257
Max  : 0.47897377610206604


In [4]:
PHI_learned = train_learned_phi(S_tr)


=== Training Learnable Φ ===
Epoch 0 Loss: 0.0052930512465536594
Epoch 20 Loss: 0.0014665769413113594
Epoch 40 Loss: 0.0006058462895452976
Epoch 60 Loss: 0.0002894380013458431
Epoch 80 Loss: 0.00017410810687579215
Epoch 100 Loss: 0.00011842712410725653
Epoch 120 Loss: 8.711894770385697e-05
Epoch 140 Loss: 6.776434747735038e-05
Epoch 160 Loss: 5.4627937061013654e-05
Epoch 180 Loss: 4.5101300202077255e-05


In [5]:
# ============================================================
# 🔥 PRINT LEARNED Φ
# ============================================================

print("\n=== LEARNED Φ (sample) ===")
print(PHI_learned[:5, :10])

print("\nStats:")
print("Mean :", PHI_learned.mean())
print("Std  :", PHI_learned.std())
print("Min  :", PHI_learned.min())
print("Max  :", PHI_learned.max())


=== LEARNED Φ (sample) ===
[[-0.05039731 -0.08363556  0.0315127   0.14979216  0.1679102  -0.0712188
   0.03564237  0.03776371 -0.18525843 -0.01080695]
 [-0.05097513 -0.01581373 -0.05786397  0.02153454 -0.02546722  0.12173908
   0.07124051  0.09784908  0.02184311  0.00945385]
 [ 0.01664166  0.01197437 -0.08334697  0.05354454  0.01394464 -0.11276968
   0.10379215 -0.07069489  0.21023247 -0.18098848]
 [ 0.05168387 -0.03670689 -0.0281046   0.02791927  0.21291403 -0.10170995
  -0.0988853   0.30094093 -0.20534344  0.12134799]
 [-0.04830992 -0.09408882 -0.00341326 -0.0408913   0.01815332 -0.00659048
   0.03805311 -0.1252595   0.02736167  0.0202538 ]]

Stats:
Mean : 0.00033842877
Std  : 0.12350542
Min  : -0.5115695
Max  : 0.4531995


In [6]:
# ============================================================
# 🔥 Φ COMPARISON
# ============================================================

diff = PHI_learned - PHI_rand

print("\n=== Φ CHANGE ===")
print("Mean difference:", np.mean(np.abs(diff)))
print("Frobenius norm :", np.linalg.norm(diff))


=== Φ CHANGE ===
Mean difference: 0.13898792853453595
Frobenius norm : 30.6046979738072


In [16]:
# ============================================================
# 🔥 FINAL CLEAN PIPELINE (ALL BUGS FIXED)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time

# ============================================================
# 🔥 FIX LABEL RANGE (CRITICAL)
# ============================================================

# Ensure labels are within [0, SZ-1]
U_tr = np.clip(U_tr, 0, SZ-1)

print("Label range:", U_tr.min(), "to", U_tr.max(), "| SZ =", SZ)


# ============================================================
# 🔥 INPUT SIZE FIX
# ============================================================

input_size = R_tr.shape[-1]


# ============================================================
# 🔥 TRAIN FUNCTION
# ============================================================

def train_model(model, name, epochs=200):

    model.train()
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    best_nmse = 1e9
    best_state = None

    print(f"\nTraining {name}...")

    for ep in range(epochs):

        total_loss = 0

        for Rb, Ub in dl:
            Rb = Rb.to(DEVICE)
            Ub = Ub.to(DEVICE)

            logits, _ = model(Rb)

            loss = loss_fn(logits.reshape(-1, SZ), Ub.reshape(-1))

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()

        # validation
        if (ep+1) % 50 == 0:
            model.eval()

            preds = np.array([
                sfdlcs_v2(Y_te[i], PHI, model, dec_prob[:SZ], K)
                for i in range(50)
            ])

            val_nmse = nmse(S_te[:50], preds)

            print(f"Epoch {ep+1} | loss={total_loss:.3f} | NMSE={val_nmse:.4f} | best={best_nmse:.4f}")

            if val_nmse < best_nmse:
                best_nmse = val_nmse
                best_state = model.state_dict()

            model.train()

    if best_state:
        model.load_state_dict(best_state)

    return model


# ============================================================
# 🔥 LSTM
# ============================================================

class SearchNetLSTM(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.lstm = nn.LSTM(input_size, 512, batch_first=True)
        self.head = nn.Linear(512, SZ)

    def forward(self, x, h=None):
        out, _ = self.lstm(x)
        return self.head(out), None


# ============================================================
# 🔥 TRANSFORMER
# ============================================================

class SearchNetTransformer(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.input_proj = nn.Linear(input_size, 256)
        self.pos_emb = nn.Embedding(1000, 256)

        enc = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=8,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )

        self.tf = nn.TransformerEncoder(enc, num_layers=3)
        self.norm = nn.LayerNorm(256)

        self.head = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, SZ)
        )

    def forward(self, x, h=None):
        B, T, _ = x.shape

        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.input_proj(x) + self.pos_emb(pos)

        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        out = self.tf(x, mask=mask)
        out = self.norm(out)

        return self.head(out), None


# ============================================================
# 🔥 OMP
# ============================================================

def omp(y):
    r = y.copy()
    sel = []

    for _ in range(K):
        corr = np.abs(PHI.T @ r)
        idx = int(np.argmax(corr))

        if idx in sel:
            break

        sel.append(idx)

        A = PHI[:, sel]
        theta = np.linalg.lstsq(A, y, rcond=None)[0]
        r = y - A @ theta

    s_hat = np.zeros(N)

    if sel:
        A = PHI[:, sel]
        theta = np.linalg.lstsq(A, y, rcond=None)[0]
        for i, idx in enumerate(sel):
            s_hat[idx] = theta[i]

    return s_hat


# ============================================================
# 🔥 TRAIN
# ============================================================

net_lstm = train_model(SearchNetLSTM(input_size).to(DEVICE), "LSTM")
net_tf   = train_model(SearchNetTransformer(input_size).to(DEVICE), "Transformer")


# ============================================================
# 🔥 INFERENCE
# ============================================================

print("\n=== EVALUATION ===")

# LSTM
t0 = time.time()
SF_lstm = np.array([
    sfdlcs_v2(Y_te[i], PHI, net_lstm, dec_prob[:SZ], K)
    for i in range(N_TEST)
])
t_lstm = (time.time() - t0) / N_TEST

# Transformer
t0 = time.time()
SF_tf = np.array([
    sfdlcs_v2(Y_te[i], PHI, net_tf, dec_prob[:SZ], K)
    for i in range(N_TEST)
])
t_tf = (time.time() - t0) / N_TEST

# OMP
t0 = time.time()
OMP_preds = np.array([omp(Y_te[i]) for i in range(N_TEST)])
t_omp = (time.time() - t0) / N_TEST


# ============================================================
# 🔥 METRICS
# ============================================================

lstm_nmse, lstm_crr = nmse(S_te, SF_lstm), crr(S_te, SF_lstm)
tf_nmse, tf_crr     = nmse(S_te, SF_tf), crr(S_te, SF_tf)
omp_nmse, omp_crr   = nmse(S_te, OMP_preds), crr(S_te, OMP_preds)


# ============================================================
# 🔥 RESULTS
# ============================================================

print("\n=== FINAL COMPARISON ===")
print("Model        NMSE       CRR        Time/sample")
print("--------------------------------------------------")
print(f"LSTM        {lstm_nmse:.4f}    {lstm_crr:.4f}    {t_lstm:.5f}s")
print(f"Transformer {tf_nmse:.4f}    {tf_crr:.4f}    {t_tf:.5f}s")
print(f"OMP         {omp_nmse:.4f}    {omp_crr:.4f}    {t_omp:.5f}s")


# ============================================================
# 🔥 IMPROVEMENT
# ============================================================

print("\n=== TRANSFORMER vs LSTM ===")
print(f"NMSE Improvement: {(lstm_nmse - tf_nmse)/lstm_nmse*100:.2f}%")
print(f"CRR  Improvement: {(tf_crr - lstm_crr)/lstm_crr*100:.2f}%")
print(f"Time Change     : {(t_tf - t_lstm)/t_lstm*100:.2f}%")

Label range: 0 to 79 | SZ = 80


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [10]:
# ============================================================
# FINAL: SFDLCS vs OMP + LEARNABLE Φ (CLEAN + CORRECT)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ================= CONFIG =================
N=480; SZ=230; K=15; M=144
SNR=12; N_TRAIN=2000; N_TEST=200; BSZ=32
DEC_THR = 0.1

# ================= DATA =================
def make_signals(n, seed):
    rng = np.random.default_rng(seed)
    S = np.zeros((n, N), np.float32)
    for i in range(n):
        idx = rng.choice(SZ, K, replace=False)
        S[i, idx] = rng.standard_normal(K)
    S /= (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
    return S

def add_noise(Y, snr_db):
    sp = (Y**2).mean(1, keepdims=True)
    std = np.sqrt(sp / 10**(snr_db/10))
    return Y + np.random.randn(*Y.shape).astype(np.float32) * std

PHI_rand = np.random.randn(M, N).astype(np.float32) / np.sqrt(M)

S_tr = make_signals(N_TRAIN, 1)
Y_tr = add_noise((PHI_rand @ S_tr.T).T, SNR)

S_te = make_signals(N_TEST, 3)
Y_te = add_noise((PHI_rand @ S_te.T).T, SNR)

# ================= LEARNABLE Φ =================

class PhiNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.phi = nn.Parameter(torch.randn(M, N) / np.sqrt(M))
    def forward(self, x):
        return x @ self.phi.T

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(M, 256),
            nn.ReLU(),
            nn.Linear(256, N)
        )
    def forward(self, y):
        return self.net(y)

phi_net = PhiNet().to(DEVICE)
decoder = Decoder().to(DEVICE)

opt_phi = torch.optim.Adam(
    list(phi_net.parameters()) + list(decoder.parameters()),
    lr=1e-3
)

print("\n=== TRAINING LEARNABLE Φ ===")

for epoch in range(20):
    total_loss = 0
    for i in range(0, N_TRAIN, BSZ):
        x = torch.tensor(S_tr[i:i+BSZ], dtype=torch.float32).to(DEVICE)
        y = phi_net(x)
        x_hat = decoder(y)
        loss = torch.mean((x_hat - x)**2)

        opt_phi.zero_grad()
        loss.backward()
        opt_phi.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss {total_loss:.4f}")

PHI_learned = phi_net.phi.detach().cpu().numpy()
Y_te_learned = (PHI_learned @ S_te.T).T

# ================= SEARCH NET =================

def make_search_data(S):
    n = S.shape[0]
    R = np.zeros((n, K, M), np.float32)
    U = np.zeros((n, K), np.int64)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=PHI_rand@h; r=y.copy(); sel=[]
        for k in range(K):
            R[z,k]=r
            idx=int(np.argmax(np.abs(s)))
            U[z,k]=idx
            s[idx]=0
            sel.append(idx)
            r=y-PHI_rand[:,sel]@h[sel]
    return R,U

R_tr, U_tr = make_search_data(S_tr)

class SearchDS(Dataset):
    def __init__(self,R,U):
        self.R=torch.tensor(R)
        self.U=torch.tensor(U)
    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i], self.U[i]

class LSTMCell(nn.Module):
    def __init__(self,I,H):
        super().__init__()
        self.Wxi=nn.Linear(I,H)
        self.Whi=nn.Linear(H,H,bias=False)
        self.Wxg=nn.Linear(I,H)
        self.Whg=nn.Linear(H,H,bias=False)
        self.Wxf=nn.Linear(I,H)
        self.Whf=nn.Linear(H,H,bias=False)
        self.Wxo=nn.Linear(I,H)
        self.Who=nn.Linear(H,H,bias=False)

    def forward(self,x,h,c):
        i=torch.sigmoid(self.Wxi(x)+self.Whi(h))
        g=torch.tanh(self.Wxg(x)+self.Whg(h))
        f=torch.sigmoid(self.Wxf(x)+self.Whf(h))
        o=torch.sigmoid(self.Wxo(x)+self.Who(h))
        c=f*c+i*g
        h=o*torch.tanh(c)
        return h,c

class SearchNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.H=512
        self.cell=LSTMCell(M,self.H)
        self.out=nn.Linear(self.H,N,bias=False)

    def forward(self,x):
        B,T,_=x.shape
        h=torch.zeros(B,self.H,device=x.device)
        c=torch.zeros(B,self.H,device=x.device)
        outs=[]
        for t in range(T):
            h,c=self.cell(x[:,t],h,c)
            outs.append(self.out(h).unsqueeze(1))
        return torch.cat(outs,dim=1)

search_net=SearchNet().to(DEVICE)

dl=DataLoader(SearchDS(R_tr,U_tr),BSZ,shuffle=True)
opt_s=optim.Adam(search_net.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()

print("\n=== TRAINING SEARCH NET ===")
for ep in range(20):
    for Rb,Ub in dl:
        Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
        logits=search_net(Rb)
        loss=loss_fn(logits.reshape(-1,N),Ub.reshape(-1))
        opt_s.zero_grad(); loss.backward(); opt_s.step()
    print("epoch",ep,"loss",loss.item())

search_net.eval()

dec_prob = (S_tr!=0).mean(0)

# ================= FIXED FUNCTIONS =================

def sfdlcs(y):
    global PHI
    r=y.copy(); hist=[]; sel=[]
    for k in range(K):
        hist.append(r.copy())
        seq=torch.tensor(np.stack(hist),dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logits=search_net(seq)[0,-1]

        probs=torch.softmax(logits,dim=-1).cpu().numpy()
        order=np.argsort(probs)[::-1]

        for cand in order:
            if cand not in sel and dec_prob[cand]>DEC_THR:
                sel.append(cand)
                break

        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta

    s_hat=np.zeros(N)
    if sel:
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat


def omp(y):
    global PHI
    r=y.copy(); sel=[]
    for _ in range(K):
        idx=int(np.argmax(np.abs(PHI.T@r)))
        if idx in sel: break
        sel.append(idx)
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta

    s_hat=np.zeros(N)
    for i,idx in enumerate(sel):
        s_hat[idx]=theta[i]
    return s_hat

# ================= METRICS =================
def nmse(t,p):
    return np.mean(np.linalg.norm(t-p,axis=1)**2 /
                   (np.linalg.norm(t,axis=1)**2+1e-12))

def crr(t,p):
    vals=[]
    for s,s_hat in zip(t,p):
        nz=np.where(s!=0)[0]
        if len(nz)==0: continue
        vals.append(np.mean(np.abs(s[nz]-s_hat[nz])<0.1))
    return np.mean(vals)

# ============================================================
# ================= YOUR BLOCK (UNCHANGED) ===================
# ============================================================

# ---------- RANDOM Φ ----------
PHI = PHI_rand

print("\n=== COMPARISON: RANDOM Φ ===")

t0 = time.time()
sf_preds = np.array([sfdlcs(Y_te[i]) for i in range(N_TEST)])
t_sf = (time.time() - t0) / N_TEST

t0 = time.time()
omp_preds = np.array([omp(Y_te[i]) for i in range(N_TEST)])
t_omp = (time.time() - t0) / N_TEST

sf_nmse = nmse(S_te, sf_preds)
sf_crr  = crr(S_te, sf_preds)

omp_nmse = nmse(S_te, omp_preds)
omp_crr  = crr(S_te, omp_preds)

print("\nAlgorithm     NMSE       CRR        Time/sample")
print("--------------------------------------------------")
print(f"SFDLCS     {sf_nmse:.4f}    {sf_crr:.4f}    {t_sf:.5f}s")
print(f"OMP        {omp_nmse:.4f}    {omp_crr:.4f}    {t_omp:.5f}s")

# ---------- LEARNED Φ ----------
PHI = PHI_learned

print("\n=== COMPARISON: LEARNED Φ ===")

t0 = time.time()
sf_preds = np.array([sfdlcs(Y_te_learned[i]) for i in range(N_TEST)])
t_sf = (time.time() - t0) / N_TEST

t0 = time.time()
omp_preds = np.array([omp(Y_te_learned[i]) for i in range(N_TEST)])
t_omp = (time.time() - t0) / N_TEST

sf_nmse = nmse(S_te, sf_preds)
sf_crr  = crr(S_te, sf_preds)

omp_nmse = nmse(S_te, omp_preds)
omp_crr  = crr(S_te, omp_preds)

print("\nAlgorithm     NMSE       CRR        Time/sample")
print("--------------------------------------------------")
print(f"SFDLCS     {sf_nmse:.4f}    {sf_crr:.4f}    {t_sf:.5f}s")
print(f"OMP        {omp_nmse:.4f}    {omp_crr:.4f}    {t_omp:.5f}s")


=== TRAINING LEARNABLE Φ ===
Epoch 0 Loss 0.1396
Epoch 1 Loss 0.1084
Epoch 2 Loss 0.0852
Epoch 3 Loss 0.0668
Epoch 4 Loss 0.0550
Epoch 5 Loss 0.0478
Epoch 6 Loss 0.0436
Epoch 7 Loss 0.0413
Epoch 8 Loss 0.0399
Epoch 9 Loss 0.0390
Epoch 10 Loss 0.0382
Epoch 11 Loss 0.0375
Epoch 12 Loss 0.0370
Epoch 13 Loss 0.0366
Epoch 14 Loss 0.0362
Epoch 15 Loss 0.0359
Epoch 16 Loss 0.0357
Epoch 17 Loss 0.0354
Epoch 18 Loss 0.0352
Epoch 19 Loss 0.0350

=== TRAINING SEARCH NET ===
epoch 0 loss 5.4921040534973145
epoch 1 loss 5.475584030151367
epoch 2 loss 5.471186637878418
epoch 3 loss 5.453254222869873
epoch 4 loss 5.463125705718994
epoch 5 loss 5.434938430786133
epoch 6 loss 5.4455246925354
epoch 7 loss 5.4420366287231445
epoch 8 loss 5.426732540130615
epoch 9 loss 5.429516792297363
epoch 10 loss 5.401615619659424
epoch 11 loss 5.375036239624023
epoch 12 loss 5.405558109283447
epoch 13 loss 5.325081825256348
epoch 14 loss 5.332470893859863
epoch 15 loss 5.346336841583252
epoch 16 loss 5.3378462791442

In [8]:
# ============================================================
# SFDLCS vs OMP + LEARNABLE Φ (CORRECT VERSION)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import time

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ================= CONFIG =================
N=480; SZ=230; K=15; M=144
SNR=12; N_TRAIN=2000; N_TEST=200; BSZ=32
DEC_THR = 0.1

# ================= DATA =================
def make_signals(n, seed):
    rng = np.random.default_rng(seed)
    S = np.zeros((n, N), np.float32)
    for i in range(n):
        idx = rng.choice(SZ, K, replace=False)
        S[i, idx] = rng.standard_normal(K)
    S /= (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
    return S

def add_noise(Y, snr_db):
    sp = (Y**2).mean(1, keepdims=True)
    std = np.sqrt(sp / 10**(snr_db/10))
    return Y + np.random.randn(*Y.shape).astype(np.float32) * std

PHI_rand = np.random.randn(M, N).astype(np.float32) / np.sqrt(M)

S_tr = make_signals(N_TRAIN, 1)
Y_tr = add_noise((PHI_rand @ S_tr.T).T, SNR)

S_te = make_signals(N_TEST, 3)
Y_te = add_noise((PHI_rand @ S_te.T).T, SNR)

# ================= LEARNABLE Φ TRAINING =================

class PhiNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.phi = nn.Parameter(torch.randn(M, N) / np.sqrt(M))

    def forward(self, x):
        return x @ self.phi.T

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(M, 256),
            nn.ReLU(),
            nn.Linear(256, N)
        )
    def forward(self, y):
        return self.net(y)

phi_net = PhiNet().to(DEVICE)
decoder = Decoder().to(DEVICE)

opt = optim.Adam(list(phi_net.parameters()) + list(decoder.parameters()), lr=1e-3)

print("\n=== TRAINING LEARNABLE Φ ===")

for epoch in range(30):
    perm = np.random.permutation(N_TRAIN)
    total_loss = 0

    for i in range(0, N_TRAIN, BSZ):
        idx = perm[i:i+BSZ]
        x = torch.tensor(S_tr[idx], dtype=torch.float32).to(DEVICE)

        y = phi_net(x)
        x_hat = decoder(y)

        loss = torch.mean((x_hat - x)**2)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss {total_loss:.4f}")

PHI_learned = phi_net.phi.detach().cpu().numpy()

# ================= SEARCH NET =================

def make_search_data(S, PHI):
    n = S.shape[0]
    R = np.zeros((n, K, M), np.float32)
    U = np.zeros((n, K), np.int64)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=PHI@h; r=y.copy(); sel=[]
        for k in range(K):
            R[z,k]=r
            idx=int(np.argmax(np.abs(s)))
            U[z,k]=idx
            s[idx]=0
            sel.append(idx)
            r=y-PHI[:,sel]@h[sel]
    return R,U

R_tr, U_tr = make_search_data(S_tr, PHI_rand)

class SearchDS(Dataset):
    def __init__(self,R,U):
        self.R=torch.tensor(R)
        self.U=torch.tensor(U)
    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i], self.U[i]

class LSTMCell(nn.Module):
    def __init__(self,I,H):
        super().__init__()
        self.Wxi=nn.Linear(I,H)
        self.Whi=nn.Linear(H,H,bias=False)
        self.Wxg=nn.Linear(I,H)
        self.Whg=nn.Linear(H,H,bias=False)
        self.Wxf=nn.Linear(I,H)
        self.Whf=nn.Linear(H,H,bias=False)
        self.Wxo=nn.Linear(I,H)
        self.Who=nn.Linear(H,H,bias=False)

    def forward(self,x,h,c):
        i=torch.sigmoid(self.Wxi(x)+self.Whi(h))
        g=torch.tanh(self.Wxg(x)+self.Whg(h))
        f=torch.sigmoid(self.Wxf(x)+self.Whf(h))
        o=torch.sigmoid(self.Wxo(x)+self.Who(h))
        c=f*c+i*g
        h=o*torch.tanh(c)
        return h,c

class SearchNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.H=512
        self.cell=LSTMCell(M,self.H)
        self.out=nn.Linear(self.H,N,bias=False)

    def forward(self,x):
        B,T,_=x.shape
        h=torch.zeros(B,self.H,device=x.device)
        c=torch.zeros(B,self.H,device=x.device)
        outs=[]
        for t in range(T):
            h,c=self.cell(x[:,t],h,c)
            outs.append(self.out(h).unsqueeze(1))
        return torch.cat(outs,dim=1)

search_net=SearchNet().to(DEVICE)

dl=DataLoader(SearchDS(R_tr,U_tr),BSZ,shuffle=True)
opt_s=optim.Adam(search_net.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()

print("\n=== TRAINING SEARCH NET ===")
for ep in range(20):
    for Rb,Ub in dl:
        Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
        logits=search_net(Rb)
        loss=loss_fn(logits.reshape(-1,N),Ub.reshape(-1))
        opt_s.zero_grad(); loss.backward(); opt_s.step()
    print("epoch",ep,"loss",loss.item())

search_net.eval()

# ================= DECISION =================
dec_prob = (S_tr!=0).mean(0)

# ================= SFDLCS =================
def sfdlcs(y, PHI):
    r=y.copy(); hist=[]; sel=[]
    for k in range(K):
        hist.append(r.copy())
        seq=torch.tensor(np.stack(hist),dtype=torch.float32).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits=search_net(seq)[0,-1]

        probs=torch.softmax(logits,dim=-1).cpu().numpy()
        order=np.argsort(probs)[::-1]

        for cand in order:
            if cand not in sel and dec_prob[cand]>DEC_THR:
                sel.append(cand)
                break

        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta

    s_hat=np.zeros(N)
    if sel:
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# ================= OMP =================
def omp(y, PHI):
    r=y.copy(); sel=[]
    for _ in range(K):
        idx=int(np.argmax(np.abs(PHI.T@r)))
        if idx in sel: break
        sel.append(idx)
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta
    s_hat=np.zeros(N)
    for i,idx in enumerate(sel):
        s_hat[idx]=theta[i]
    return s_hat

# ================= EVALUATION =================
print("\n=== FINAL COMPARISON ===")

# RANDOM Φ
sf_rand = np.array([sfdlcs(Y_te[i], PHI_rand) for i in range(N_TEST)])
omp_rand = np.array([omp(Y_te[i], PHI_rand) for i in range(N_TEST)])

# LEARNED Φ
Y_te_learn = (PHI_learned @ S_te.T).T
sf_learn = np.array([sfdlcs(Y_te_learn[i], PHI_learned) for i in range(N_TEST)])

def nmse(t,p):
    return np.mean(np.linalg.norm(t-p,axis=1)**2 /
                   (np.linalg.norm(t,axis=1)**2+1e-12))

print("\nAlgorithm               NMSE")
print("-----------------------------------")
print(f"SFDLCS + Random Φ   {nmse(S_te,sf_rand):.4f}")
print(f"SFDLCS + Learned Φ  {nmse(S_te,sf_learn):.4f}   🔥")
print(f"OMP + Random Φ      {nmse(S_te,omp_rand):.4f}")


=== TRAINING LEARNABLE Φ ===
Epoch 0 Loss 0.1396
Epoch 1 Loss 0.1085
Epoch 2 Loss 0.0855
Epoch 3 Loss 0.0673
Epoch 4 Loss 0.0555
Epoch 5 Loss 0.0482
Epoch 6 Loss 0.0441
Epoch 7 Loss 0.0418
Epoch 8 Loss 0.0404
Epoch 9 Loss 0.0393
Epoch 10 Loss 0.0384
Epoch 11 Loss 0.0379
Epoch 12 Loss 0.0374
Epoch 13 Loss 0.0368
Epoch 14 Loss 0.0365
Epoch 15 Loss 0.0361
Epoch 16 Loss 0.0358
Epoch 17 Loss 0.0355
Epoch 18 Loss 0.0353
Epoch 19 Loss 0.0351
Epoch 20 Loss 0.0350
Epoch 21 Loss 0.0348
Epoch 22 Loss 0.0347
Epoch 23 Loss 0.0346
Epoch 24 Loss 0.0344
Epoch 25 Loss 0.0342
Epoch 26 Loss 0.0341
Epoch 27 Loss 0.0340
Epoch 28 Loss 0.0339
Epoch 29 Loss 0.0337

=== TRAINING SEARCH NET ===
epoch 0 loss 5.4921040534973145
epoch 1 loss 5.475584030151367
epoch 2 loss 5.471186637878418
epoch 3 loss 5.453254222869873
epoch 4 loss 5.463125705718994
epoch 5 loss 5.434938430786133
epoch 6 loss 5.4455246925354
epoch 7 loss 5.4420366287231445
epoch 8 loss 5.426732540130615
epoch 9 loss 5.429516792297363
epoch 10 lo

In [6]:
# ============================================================
# FINAL SFDLCS vs OMP (FULLY CORRECTED — ONE CELL)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ================= CONFIG =================
N=480; SZ=80; K=15; M=64
SNR=12; N_TRAIN=2000; N_TEST=200; BSZ=32
DEC_THR = 0.1

# ================= DATA =================
def make_uwb_signals(n, seed):
    rng = np.random.default_rng(seed)
    S = np.zeros((n, N), np.float32)
    for i in range(n):
        centres = rng.choice(SZ - 6, K, replace=False) + 3
        for c in centres:
            amp = rng.standard_normal()
            width = rng.uniform(1.5, 3.0)
            for d in range(-4, 5):
                if 0 <= c+d < N:
                    S[i, c+d] += amp * np.exp(-0.5*(d/width)**2)
    S /= (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
    return S

def add_noise(Y, snr_db):
    sp = (Y**2).mean(1, keepdims=True)
    std = np.sqrt(sp / 10**(snr_db/10))
    return Y + np.random.randn(*Y.shape).astype(np.float32) * std

PHI = np.random.randn(M, N).astype(np.float32) / np.sqrt(M)

S_tr = make_uwb_signals(N_TRAIN, 1)
Y_tr = add_noise((PHI @ S_tr.T).T, SNR)

S_te = make_uwb_signals(N_TEST, 3)
Y_te = add_noise((PHI @ S_te.T).T, SNR)

# ================= SEARCH DATA (FIXED) =================
def make_search_data(S, noise_std=0.01):
    n = S.shape[0]
    R = np.zeros((n, K, M), np.float32)
    U = np.zeros((n, K), np.int64)
    rng = np.random.default_rng(42)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=PHI@h; r=y.copy(); sel=[]

        for k in range(K):
            # 🔥 KEY FIX: noisy residual
            r_noisy = r + noise_std * rng.standard_normal(r.shape).astype(np.float32)
            R[z,k]=r_noisy

            idx=int(np.argmax(np.abs(s[:SZ])))
            U[z,k]=idx

            s[idx]=0
            sel.append(idx)

            r=y-PHI[:,sel]@h[sel]

    return R,U

R_tr, U_tr = make_search_data(S_tr)

class SearchDS(Dataset):
    def __init__(self,R,U):
        self.R=torch.tensor(R)
        self.U=torch.tensor(U)
    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i], self.U[i]

# ================= LSTM =================
class LSTMCell(nn.Module):
    def __init__(self,I,H):
        super().__init__()
        self.Wxi=nn.Linear(I,H)
        self.Whi=nn.Linear(H,H,bias=False)
        self.Wxg=nn.Linear(I,H)
        self.Whg=nn.Linear(H,H,bias=False)
        self.Wxf=nn.Linear(I,H)
        self.Whf=nn.Linear(H,H,bias=False)
        self.Wxo=nn.Linear(I,H)
        self.Who=nn.Linear(H,H,bias=False)

    def forward(self,x,h,c):
        i=torch.sigmoid(self.Wxi(x)+self.Whi(h))
        g=torch.tanh(self.Wxg(x)+self.Whg(h))
        f=torch.sigmoid(self.Wxf(x)+self.Whf(h))
        o=torch.sigmoid(self.Wxo(x)+self.Who(h))
        c=f*c+i*g
        h=o*torch.tanh(c)
        return h,c

class SearchNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.H=512
        self.cell=LSTMCell(M,self.H)
        self.out=nn.Linear(self.H,N,bias=False)

    def forward(self,x):
        B,T,_=x.shape
        h=torch.zeros(B,self.H,device=x.device)
        c=torch.zeros(B,self.H,device=x.device)
        outs=[]
        for t in range(T):
            h,c=self.cell(x[:,t],h,c)
            outs.append(self.out(h).unsqueeze(1))
        return torch.cat(outs,dim=1)

search_net=SearchNet().to(DEVICE)

# ================= DECISION NETWORK =================
class DecisionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(N,256),
            nn.ReLU(),
            nn.Linear(256,N),
            nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)

dec_net=DecisionNet().to(DEVICE)

idx_input=torch.eye(N).to(DEVICE)
labels=torch.tensor((S_tr!=0).mean(0),dtype=torch.float32)\
        .unsqueeze(0).repeat(N,1).to(DEVICE)

opt=optim.Adam(dec_net.parameters(),lr=1e-3)

for ep in range(150):
    pred=dec_net(idx_input)
    loss=nn.BCELoss()(pred,labels)
    opt.zero_grad(); loss.backward(); opt.step()

dec_net.eval()
with torch.no_grad():
    dec_prob=dec_net(idx_input)[0].cpu().numpy()

# ================= TRAIN SEARCH =================
dl=DataLoader(SearchDS(R_tr,U_tr),BSZ,shuffle=True)
opt=optim.Adam(search_net.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()

for ep in range(40):
    for Rb,Ub in dl:
        Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
        logits=search_net(Rb)
        loss=loss_fn(logits.reshape(-1,N),Ub.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(search_net.parameters(),1.0)
        opt.step()
    print("epoch",ep,"loss",loss.item())

search_net.eval()

# ================= SFDLCS =================
def sfdlcs(y):
    r=y.copy(); hist=[]; sel=[]
    for k in range(K):
        hist.append(r.copy())
        seq=torch.tensor(np.stack(hist),dtype=torch.float32)\
            .unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits=search_net(seq)[0,-1]

        probs=torch.softmax(logits,dim=-1).cpu().numpy()
        order=np.argsort(probs)[::-1]

        chosen=None
        for cand in order:
            if cand in sel: continue
            if dec_prob[cand]>DEC_THR:
                chosen=cand; break

        if chosen is None: break
        sel.append(chosen)

        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta

    s_hat=np.zeros(N)
    if sel:
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# ================= OMP =================
def omp(y):
    r=y.copy(); sel=[]
    for _ in range(K):
        idx=int(np.argmax(np.abs(PHI.T@r)))
        if idx in sel: break
        sel.append(idx)
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta
    s_hat=np.zeros(N)
    if sel:
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# ================= METRICS =================
def nmse(t,p):
    return np.mean(np.linalg.norm(t-p,axis=1)**2 /
                   (np.linalg.norm(t,axis=1)**2+1e-12))

def crr(t,p):
    vals=[]
    for s,s_hat in zip(t,p):
        nz=np.where(s!=0)[0]
        if len(nz)==0: continue
        vals.append(np.mean(np.abs(s[nz]-s_hat[nz])<0.1))
    return np.mean(vals)

# ================= EVALUATION =================
print("\n=== FINAL COMPARISON ===")

t0=time.time()
sf_preds=np.array([sfdlcs(Y_te[i]) for i in range(N_TEST)])
t_sf=(time.time()-t0)/N_TEST

t0=time.time()
omp_preds=np.array([omp(Y_te[i]) for i in range(N_TEST)])
t_omp=(time.time()-t0)/N_TEST

print("\nAlgorithm     NMSE       CRR        Time/sample")
print("--------------------------------------------------")
print(f"SFDLCS     {nmse(S_te,sf_preds):.4f}    {crr(S_te,sf_preds):.4f}    {t_sf:.5f}s")
print(f"OMP        {nmse(S_te,omp_preds):.4f}    {crr(S_te,omp_preds):.4f}    {t_omp:.5f}s")

epoch 0 loss 4.491702079772949
epoch 1 loss 4.43855094909668
epoch 2 loss 4.4426116943359375
epoch 3 loss 4.36037015914917
epoch 4 loss 4.3907856941223145
epoch 5 loss 4.366541862487793
epoch 6 loss 4.294230937957764
epoch 7 loss 4.285674095153809
epoch 8 loss 4.13443660736084
epoch 9 loss 4.182432174682617
epoch 10 loss 4.012572765350342
epoch 11 loss 4.002829551696777
epoch 12 loss 3.961219072341919
epoch 13 loss 3.7369792461395264
epoch 14 loss 3.9616968631744385
epoch 15 loss 3.718100070953369
epoch 16 loss 3.7262933254241943
epoch 17 loss 3.7006750106811523
epoch 18 loss 3.5989840030670166
epoch 19 loss 3.469672918319702
epoch 20 loss 3.28542423248291
epoch 21 loss 3.3044726848602295
epoch 22 loss 3.196126699447632
epoch 23 loss 3.0704314708709717
epoch 24 loss 3.0689361095428467
epoch 25 loss 2.9785828590393066
epoch 26 loss 2.913872003555298
epoch 27 loss 2.6450002193450928
epoch 28 loss 2.810478687286377
epoch 29 loss 2.6831164360046387
epoch 30 loss 2.736034393310547
epoch 31 

In [7]:
# ============================================================
# FINAL SFDLCS vs OMP (THEORY-CORRECT VERSION)
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import time

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ================= CONFIG =================
N=480; SZ=230; K=15; M=144   # ← IMPORTANT FIX (M>K, realistic)
SNR=12; N_TRAIN=2000; N_TEST=200; BSZ=32
DEC_THR = 0.1

# ================= DATA (TRUE SPARSE) =================
def make_signals(n, seed):
    rng = np.random.default_rng(seed)
    S = np.zeros((n, N), np.float32)

    for i in range(n):
        idx = rng.choice(SZ, K, replace=False)
        S[i, idx] = rng.standard_normal(K)

    S /= (np.linalg.norm(S, axis=1, keepdims=True) + 1e-9)
    return S

def add_noise(Y, snr_db):
    sp = (Y**2).mean(1, keepdims=True)
    std = np.sqrt(sp / 10**(snr_db/10))
    return Y + np.random.randn(*Y.shape).astype(np.float32) * std

PHI = np.random.randn(M, N).astype(np.float32) / np.sqrt(M)

S_tr = make_signals(N_TRAIN, 1)
Y_tr = add_noise((PHI @ S_tr.T).T, SNR)

S_te = make_signals(N_TEST, 3)
Y_te = add_noise((PHI @ S_te.T).T, SNR)

# ================= SEARCH DATA =================
def make_search_data(S, noise_std=0.01):
    n = S.shape[0]
    R = np.zeros((n, K, M), np.float32)
    U = np.zeros((n, K), np.int64)
    rng = np.random.default_rng(42)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=PHI@h; r=y.copy(); sel=[]

        for k in range(K):
            r_noisy = r + noise_std * rng.standard_normal(r.shape).astype(np.float32)
            R[z,k]=r_noisy

            idx=int(np.argmax(np.abs(s)))
            U[z,k]=idx

            s[idx]=0
            sel.append(idx)

            r=y-PHI[:,sel]@h[sel]

    return R,U

R_tr, U_tr = make_search_data(S_tr)

class SearchDS(Dataset):
    def __init__(self,R,U):
        self.R=torch.tensor(R)
        self.U=torch.tensor(U)
    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i], self.U[i]

# ================= LSTM =================
class LSTMCell(nn.Module):
    def __init__(self,I,H):
        super().__init__()
        self.Wxi=nn.Linear(I,H)
        self.Whi=nn.Linear(H,H,bias=False)
        self.Wxg=nn.Linear(I,H)
        self.Whg=nn.Linear(H,H,bias=False)
        self.Wxf=nn.Linear(I,H)
        self.Whf=nn.Linear(H,H,bias=False)
        self.Wxo=nn.Linear(I,H)
        self.Who=nn.Linear(H,H,bias=False)

    def forward(self,x,h,c):
        i=torch.sigmoid(self.Wxi(x)+self.Whi(h))
        g=torch.tanh(self.Wxg(x)+self.Whg(h))
        f=torch.sigmoid(self.Wxf(x)+self.Whf(h))
        o=torch.sigmoid(self.Wxo(x)+self.Who(h))
        c=f*c+i*g
        h=o*torch.tanh(c)
        return h,c

class SearchNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.H=512
        self.cell=LSTMCell(M,self.H)
        self.out=nn.Linear(self.H,N,bias=False)

    def forward(self,x):
        B,T,_=x.shape
        h=torch.zeros(B,self.H,device=x.device)
        c=torch.zeros(B,self.H,device=x.device)
        outs=[]
        for t in range(T):
            h,c=self.cell(x[:,t],h,c)
            outs.append(self.out(h).unsqueeze(1))
        return torch.cat(outs,dim=1)

search_net=SearchNet().to(DEVICE)

# ================= DECISION NET =================
class DecisionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(N,256),
            nn.ReLU(),
            nn.Linear(256,N),
            nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)

dec_net=DecisionNet().to(DEVICE)

idx_input=torch.eye(N).to(DEVICE)
labels=torch.tensor((S_tr!=0).mean(0),dtype=torch.float32)\
        .unsqueeze(0).repeat(N,1).to(DEVICE)

opt=optim.Adam(dec_net.parameters(),lr=1e-3)

for ep in range(100):
    pred=dec_net(idx_input)
    loss=nn.BCELoss()(pred,labels)
    opt.zero_grad(); loss.backward(); opt.step()

dec_net.eval()
with torch.no_grad():
    dec_prob=dec_net(idx_input)[0].cpu().numpy()

# ================= TRAIN SEARCH =================
dl=DataLoader(SearchDS(R_tr,U_tr),BSZ,shuffle=True)
opt=optim.Adam(search_net.parameters(),lr=1e-3)
loss_fn=nn.CrossEntropyLoss()

for ep in range(30):
    for Rb,Ub in dl:
        Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
        logits=search_net(Rb)
        loss=loss_fn(logits.reshape(-1,N),Ub.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(search_net.parameters(),1.0)
        opt.step()
    print("epoch",ep,"loss",loss.item())

search_net.eval()

# ================= SFDLCS =================
def sfdlcs(y):
    r=y.copy(); hist=[]; sel=[]
    for k in range(K):
        hist.append(r.copy())
        seq=torch.tensor(np.stack(hist),dtype=torch.float32)\
            .unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits=search_net(seq)[0,-1]

        probs=torch.softmax(logits,dim=-1).cpu().numpy()
        order=np.argsort(probs)[::-1]

        chosen=None
        for cand in order:
            if cand in sel: continue
            if dec_prob[cand]>DEC_THR:
                chosen=cand; break

        if chosen is None: break
        sel.append(chosen)

        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta

    s_hat=np.zeros(N)
    if sel:
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# ================= OMP =================
def omp(y):
    r=y.copy(); sel=[]
    for _ in range(K):
        idx=int(np.argmax(np.abs(PHI.T@r)))
        if idx in sel: break
        sel.append(idx)
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        r=y-A@theta
    s_hat=np.zeros(N)
    if sel:
        A=PHI[:,sel]
        theta=np.linalg.lstsq(A,y,rcond=None)[0]
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# ================= METRICS =================
def nmse(t,p):
    return np.mean(np.linalg.norm(t-p,axis=1)**2 /
                   (np.linalg.norm(t,axis=1)**2+1e-12))

def crr(t,p):
    vals=[]
    for s,s_hat in zip(t,p):
        nz=np.where(s!=0)[0]
        if len(nz)==0: continue
        vals.append(np.mean(np.abs(s[nz]-s_hat[nz])<0.1))
    return np.mean(vals)

# ================= EVALUATION =================
print("\n=== FINAL COMPARISON ===")

t0=time.time()
sf_preds=np.array([sfdlcs(Y_te[i]) for i in range(N_TEST)])
t_sf=(time.time()-t0)/N_TEST

t0=time.time()
omp_preds=np.array([omp(Y_te[i]) for i in range(N_TEST)])
t_omp=(time.time()-t0)/N_TEST

print("\nAlgorithm     NMSE       CRR        Time/sample")
print("--------------------------------------------------")
print(f"SFDLCS     {nmse(S_te,sf_preds):.4f}    {crr(S_te,sf_preds):.4f}    {t_sf:.5f}s")
print(f"OMP        {nmse(S_te,omp_preds):.4f}    {crr(S_te,omp_preds):.4f}    {t_omp:.5f}s")

epoch 0 loss 5.506138801574707
epoch 1 loss 5.494892597198486
epoch 2 loss 5.486929893493652
epoch 3 loss 5.450493812561035
epoch 4 loss 5.452913284301758
epoch 5 loss 5.425946235656738
epoch 6 loss 5.441801071166992
epoch 7 loss 5.4440226554870605
epoch 8 loss 5.421424388885498
epoch 9 loss 5.432981491088867
epoch 10 loss 5.420830249786377
epoch 11 loss 5.420226573944092
epoch 12 loss 5.4134602546691895
epoch 13 loss 5.370540142059326
epoch 14 loss 5.373927593231201
epoch 15 loss 5.256712436676025
epoch 16 loss 5.299014568328857
epoch 17 loss 5.1167683601379395
epoch 18 loss 5.159598350524902
epoch 19 loss 4.877253532409668
epoch 20 loss 4.751075744628906
epoch 21 loss 4.629605293273926
epoch 22 loss 4.594901084899902
epoch 23 loss 4.290968894958496
epoch 24 loss 4.0845770835876465
epoch 25 loss 4.136303901672363
epoch 26 loss 3.64180850982666
epoch 27 loss 3.3469157218933105
epoch 28 loss 3.384141206741333
epoch 29 loss 3.2070913314819336

=== FINAL COMPARISON ===

Algorithm     NMSE

In [5]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   LEARN Φ FOR OMP / SFDLCS (CORRECT VERSION)                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n=== TRAINING PROPER LEARNABLE Φ (OMP-COMPATIBLE) ===")

class PhiOptim(nn.Module):
    def __init__(self):
        super().__init__()
        self.Phi = nn.Parameter(torch.randn(M,N)/np.sqrt(M))

    def forward(self, x):
        return x @ self.Phi.T

phi_model = PhiOptim().to(DEVICE)
opt_phi = optim.Adam(phi_model.parameters(), lr=1e-3)

LAMBDA = 0.1   # coherence regularization

def omp_torch(y, Phi, K=15):
    # differentiable approximation (soft selection)
    x_hat = torch.zeros(N, device=y.device)

    r = y.clone()
    Phi_t = Phi.t()

    for _ in range(K):
        corr = torch.abs(Phi_t @ r)
        idx = torch.argmax(corr)   # still non-diff but ok for training stability

        a = Phi[:, idx]
        coef = (a @ r) / (a @ a + 1e-8)

        x_hat[idx] += coef
        r = r - coef * a

    return x_hat

X = torch.tensor(S_tr, dtype=torch.float32).to(DEVICE)

for ep in range(50):
    opt_phi.zero_grad()

    Phi = phi_model.Phi

    # forward
    Y = X @ Phi.T

    recon = []
    for i in range(100):  # subset for speed
        y = Y[i]
        x_hat = omp_torch(y, Phi)
        recon.append(x_hat)

    recon = torch.stack(recon)

    # reconstruction loss
    loss_rec = ((recon - X[:100])**2).mean()

    # coherence loss
    I = torch.eye(N, device=DEVICE)
    gram = Phi.T @ Phi
    loss_coh = ((gram - I)**2).mean()

    loss = loss_rec + LAMBDA * loss_coh

    loss.backward()
    opt_phi.step()

    if ep % 10 == 0:
        print(f"Epoch {ep} | Rec {loss_rec.item():.4f} | Coh {loss_coh.item():.4f}")

Phi_learn = phi_model.Phi.detach().cpu().numpy()

print("\n✅ Learned Φ ready (OMP/SFDLCS compatible)")


=== TRAINING PROPER LEARNABLE Φ (OMP-COMPATIBLE) ===
Epoch 0 | Rec 0.0005 | Coh 0.0157
Epoch 10 | Rec 0.0003 | Coh 0.0122
Epoch 20 | Rec 0.0002 | Coh 0.0096
Epoch 30 | Rec 0.0001 | Coh 0.0077
Epoch 40 | Rec 0.0001 | Coh 0.0063

✅ Learned Φ ready (OMP/SFDLCS compatible)


In [6]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   CELL 3 — FINAL EVALUATION (PROPER Φ)                              ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n=== CELL 3: EVALUATING PROPER LEARNED Φ ===")

Phi_rand = PHI.copy()   # original
# Phi_learn already computed

# ---------- OMP with custom Φ ----------
def omp_phi(y, Phi):
    r=y.copy(); sel=[]
    for _ in range(K_MAX):
        idx=int(np.argmax(np.abs(Phi.T@r)))
        if idx in sel: break
        sel.append(idx)
        A=Phi[:,sel]
        theta,_ ,_,_=np.linalg.lstsq(A,y,rcond=None)
        r=y-A@theta
    s=np.zeros(N)
    for i,idx in enumerate(sel): s[idx]=theta[i]
    return s

# ---------- SFDLCS with custom Φ ----------
def sfdlcs_phi(y, Phi):
    r=y.copy(); sel=[]
    for k in range(K_MAX):

        x=torch.from_numpy(r).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits=net(x)[0,0]
        probs=torch.softmax(logits,dim=-1).cpu().detach().numpy()

        order=np.argsort(probs)[::-1]
        chosen=None

        for cand in order:
            if cand in sel: continue
            if dec_prob[cand] > DEC_THR:
                chosen=cand
                break

        if chosen is None:
            break

        sel.append(chosen)

        A=Phi[:,sel]
        theta,_ ,_,_=np.linalg.lstsq(A,y,rcond=None)
        r=y-A@theta

        if np.linalg.norm(r)<1e-4:
            break

    s_hat=np.zeros(N)
    if sel:
        A=Phi[:,sel]
        theta,_ ,_,_=np.linalg.lstsq(A,y,rcond=None)
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# ---------- evaluation ----------
def evaluate(Phi, method, name):
    print(f"\n{name}")
    t=time.time()

    recon=[]
    for i in range(N_TEST):
        y = Phi @ S_te[i]
        recon.append(method(y,Phi))

    recon=np.array(recon)
    t=(time.time()-t)/N_TEST

    n = nmse(S_te,recon)
    c = crr(S_te,recon)

    print(f"{name} NMSE={n:.4f} CRR={c:.4f} Time={t:.5f}")
    return n,c,t

print("\n=== COMPARISON (FINAL) ===")

res3 = {
    "OMP Random Φ": evaluate(Phi_rand, omp_phi, "OMP Random Φ"),
    "OMP Learned Φ": evaluate(Phi_learn, omp_phi, "OMP Learned Φ"),
    "SFDLCS Random Φ": evaluate(Phi_rand, sfdlcs_phi, "SFDLCS Random Φ"),
    "SFDLCS Learned Φ": evaluate(Phi_learn, sfdlcs_phi, "SFDLCS Learned Φ"),
}

print("\n=== FINAL RESULTS (CELL 3) ===")
for k,v in res3.items():
    print(f"{k:20s} NMSE={v[0]:.4f}  CRR={v[1]:.4f}  Time={v[2]:.5f}")


=== CELL 3: EVALUATING PROPER LEARNED Φ ===

=== COMPARISON (FINAL) ===

OMP Random Φ
OMP Random Φ NMSE=0.1523 CRR=0.8870 Time=0.00164

OMP Learned Φ
OMP Learned Φ NMSE=0.0349 CRR=0.9707 Time=0.00168

SFDLCS Random Φ
SFDLCS Random Φ NMSE=0.6337 CRR=0.5183 Time=0.02463

SFDLCS Learned Φ
SFDLCS Learned Φ NMSE=1.0698 CRR=0.3637 Time=0.02416

=== FINAL RESULTS (CELL 3) ===
OMP Random Φ         NMSE=0.1523  CRR=0.8870  Time=0.00164
OMP Learned Φ        NMSE=0.0349  CRR=0.9707  Time=0.00168
SFDLCS Random Φ      NMSE=0.6337  CRR=0.5183  Time=0.02463
SFDLCS Learned Φ     NMSE=1.0698  CRR=0.3637  Time=0.02416


In [7]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   CELL 4 — RETRAIN SFDLCS WITH LEARNED Φ                            ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n=== CELL 4: RETRAINING SFDLCS WITH LEARNED Φ ===")

PHI_NEW = Phi_learn.copy()

# -------- regenerate training data with NEW Φ --------
def make_search_data_phi(S, Phi):
    n,_ = S.shape
    R = np.zeros((n,K_MAX,M),np.float32)
    U = np.zeros((n,K_MAX),np.int64)

    for z in range(n):
        s=S[z].copy(); h=S[z].copy()
        y=Phi@h; r=y.copy(); sel=[]
        for k in range(K_MAX):
            R[z,k]=r
            idx=int(np.argmax(np.abs(s)))
            U[z,k]=idx
            s[idx]=0; sel.append(idx)
            r=y-Phi[:,sel]@h[sel]
    return R,U

R_tr_new, U_tr_new = make_search_data_phi(S_tr, PHI_NEW)

class DS2(Dataset):
    def __init__(self,R,U):
        self.R=torch.tensor(R,dtype=torch.float32)
        self.U=torch.tensor(U,dtype=torch.long)
    def __len__(self): return len(self.R)
    def __getitem__(self,i): return self.R[i],self.U[i]

dl2 = DataLoader(DS2(R_tr_new,U_tr_new), BSZ, shuffle=True)

# -------- new search net --------
net_new = SearchNet().to(DEVICE)
opt_new = optim.Adam(net_new.parameters(), lr=1e-2)
ce = nn.CrossEntropyLoss()

print("\nTraining NEW SFDLCS (aligned with learned Φ)...")

for ep in range(50):
    total=0
    for Rb,Ub in dl2:
        Rb,Ub=Rb.to(DEVICE),Ub.to(DEVICE)
        logits=net_new(Rb)
        loss=ce(logits.view(-1,N),Ub.view(-1))
        opt_new.zero_grad(); loss.backward(); opt_new.step()
        total+=loss.item()
    print(f"Epoch {ep} Loss {total/len(dl2):.4f}")

# -------- new SFDLCS using learned Φ --------
def sfdlcs_learned(y, Phi):
    r=y.copy(); sel=[]
    for k in range(K_MAX):

        x=torch.from_numpy(r).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits=net_new(x)[0,0]
        probs=torch.softmax(logits,dim=-1).cpu().detach().numpy()

        order=np.argsort(probs)[::-1]
        chosen=None

        for cand in order:
            if cand in sel: continue
            if dec_prob[cand] > DEC_THR:
                chosen=cand
                break

        if chosen is None:
            break

        sel.append(chosen)

        A=Phi[:,sel]
        theta,_ ,_,_=np.linalg.lstsq(A,y,rcond=None)
        r=y-A@theta

        if np.linalg.norm(r)<1e-4:
            break

    s_hat=np.zeros(N)
    if sel:
        A=Phi[:,sel]
        theta,_ ,_,_=np.linalg.lstsq(A,y,rcond=None)
        for i,idx in enumerate(sel):
            s_hat[idx]=theta[i]
    return s_hat

# -------- evaluation --------
print("\n=== FINAL IMPROVED RESULT ===")

res_final = {
    "SFDLCS Learned Φ (retrained)": evaluate(PHI_NEW, sfdlcs_learned, "SFDLCS Learned Φ RETRAINED")
}

print("\n=== FINAL AFTER FIX ===")
for k,v in res_final.items():
    print(f"{k:30s} NMSE={v[0]:.4f}  CRR={v[1]:.4f}")


=== CELL 4: RETRAINING SFDLCS WITH LEARNED Φ ===

Training NEW SFDLCS (aligned with learned Φ)...
Epoch 0 Loss 3.9475
Epoch 1 Loss 3.4796
Epoch 2 Loss 3.3531
Epoch 3 Loss 3.3212
Epoch 4 Loss 3.2905
Epoch 5 Loss 3.2146
Epoch 6 Loss 3.1184
Epoch 7 Loss 2.9946
Epoch 8 Loss 2.7933
Epoch 9 Loss 2.5711
Epoch 10 Loss 2.3456
Epoch 11 Loss 2.0858
Epoch 12 Loss 1.8843
Epoch 13 Loss 1.6605
Epoch 14 Loss 1.4638
Epoch 15 Loss 1.3131
Epoch 16 Loss 1.2010
Epoch 17 Loss 1.0507
Epoch 18 Loss 0.9416
Epoch 19 Loss 0.8519
Epoch 20 Loss 0.7505
Epoch 21 Loss 0.6879
Epoch 22 Loss 0.6124
Epoch 23 Loss 0.5704
Epoch 24 Loss 0.4882
Epoch 25 Loss 0.4277
Epoch 26 Loss 0.3740
Epoch 27 Loss 0.3276
Epoch 28 Loss 0.2917
Epoch 29 Loss 0.2494
Epoch 30 Loss 0.2244
Epoch 31 Loss 0.1925
Epoch 32 Loss 0.1600
Epoch 33 Loss 0.1329
Epoch 34 Loss 0.1048
Epoch 35 Loss 0.0938
Epoch 36 Loss 0.0769
Epoch 37 Loss 0.0579
Epoch 38 Loss 0.0475
Epoch 39 Loss 0.0438
Epoch 40 Loss 0.0492
Epoch 41 Loss 0.0507
Epoch 42 Loss 0.0486
Epoch 43

In [8]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   CELL 5 — IMPROVED SFDLCS (NO HARD GATING)                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n=== CELL 5: IMPROVED SFDLCS (NO DECISION GATE) ===")

def sfdlcs_improved(y, Phi):
    r = y.copy()
    sel = []

    for k in range(K_MAX):

        # LSTM prediction
        x = torch.from_numpy(r).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits = net_new(x)[0,0]
        probs = torch.softmax(logits, dim=-1).cpu().detach().numpy()

        # take best index (no threshold filtering)
        order = np.argsort(probs)[::-1]

        chosen = None
        for cand in order:
            if cand not in sel:
                chosen = cand
                break

        if chosen is None:
            break

        sel.append(chosen)

        # LS update
        A = Phi[:, sel]
        theta, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        r = y - A @ theta

        # early stopping
        if np.linalg.norm(r) < 1e-4:
            break

    # reconstruction
    s_hat = np.zeros(N)
    if sel:
        A = Phi[:, sel]
        theta, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        for i, idx in enumerate(sel):
            s_hat[idx] = theta[i]

    return s_hat


print("\n=== EVALUATING IMPROVED SFDLCS ===")

res_improved = evaluate(Phi_learn, sfdlcs_improved, "SFDLCS IMPROVED (Learned Φ)")

print("\n=== RESULT ===")
print(f"SFDLCS Improved NMSE={res_improved[0]:.4f}  CRR={res_improved[1]:.4f}")


=== CELL 5: IMPROVED SFDLCS (NO DECISION GATE) ===

=== EVALUATING IMPROVED SFDLCS ===

SFDLCS IMPROVED (Learned Φ)
SFDLCS IMPROVED (Learned Φ) NMSE=0.1762 CRR=0.8150 Time=0.02465

=== RESULT ===
SFDLCS Improved NMSE=0.1762  CRR=0.8150


In [9]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║   CELL 6 — STRONG SFDLCS (TOP-K SEARCH)                             ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n=== CELL 6: STRONG SFDLCS (TOP-K SEARCH) ===")

TOPK = 3   # try 3 candidates each step

def sfdlcs_strong(y, Phi):
    r = y.copy()
    sel = []

    for k in range(K_MAX):

        # LSTM prediction
        x = torch.from_numpy(r).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
        logits = net_new(x)[0,0]
        probs = torch.softmax(logits, dim=-1).cpu().detach().numpy()

        order = np.argsort(probs)[::-1]

        best_idx = None
        best_residual = 1e9

        # try TOP-K candidates
        for cand in order[:TOPK]:

            if cand in sel:
                continue

            temp_sel = sel + [cand]
            A = Phi[:, temp_sel]

            theta, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
            r_temp = y - A @ theta

            res_norm = np.linalg.norm(r_temp)

            if res_norm < best_residual:
                best_residual = res_norm
                best_idx = cand

        if best_idx is None:
            break

        sel.append(best_idx)

        # update residual
        A = Phi[:, sel]
        theta, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        r = y - A @ theta

        if np.linalg.norm(r) < 1e-4:
            break

    # reconstruction
    s_hat = np.zeros(N)
    if sel:
        A = Phi[:, sel]
        theta, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
        for i, idx in enumerate(sel):
            s_hat[idx] = theta[i]

    return s_hat


print("\n=== EVALUATING STRONG SFDLCS ===")

res_strong = evaluate(Phi_learn, sfdlcs_strong, "SFDLCS STRONG (TOP-K)")

print("\n=== RESULT ===")
print(f"SFDLCS Strong NMSE={res_strong[0]:.4f}  CRR={res_strong[1]:.4f}")


=== CELL 6: STRONG SFDLCS (TOP-K SEARCH) ===

=== EVALUATING STRONG SFDLCS ===

SFDLCS STRONG (TOP-K)
SFDLCS STRONG (TOP-K) NMSE=0.1583 CRR=0.7927 Time=0.01857

=== RESULT ===
SFDLCS Strong NMSE=0.1583  CRR=0.7927
